# Fake News & Source Credibility Detector — Colab (Git-based)

This notebook pulls **all code straight from GitHub** (no Google Drive copy),
so you always run the latest version — no stale-folder traps.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then *Runtime → Run all*.

Repo: https://github.com/Sanjana132/FakeNewsCredibilityAssesor.git


## 1 · Verify the GPU


In [2]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM (GB)      :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
else:
    print('NO GPU — Runtime → Change runtime type → T4 GPU, then re-run.')


CUDA available : True
GPU            : Tesla T4
VRAM (GB)      : 15.6


## 2 · Clone the repo **fresh** from GitHub

`rm -rf` first so re-running this cell always gives you the latest code
(this is what prevents the stale-code problem). Data/model folders are
recreated by later cells, so nothing important is lost here.


In [3]:
import os
REPO = 'https://github.com/Sanjana132/FakeNewsCredibilityAssesor.git'
DIR  = '/content/FakeNewsCredibilityAssesor'
# Step OUT to /content first — otherwise re-running this cell deletes the
# directory the shell is sitting in and git clone fails with
# 'Unable to read current working directory'.
os.chdir('/content')
!rm -rf $DIR
!git clone --depth 1 $REPO $DIR
os.chdir(DIR)
print('\nNow in:', os.getcwd())
print('Latest commit:')
!git log --oneline -1
# Sanity: confirm the Phase-5 training fix is present
src = open('phase5_deberta.py').read()
print('phase5 fix present:', all(k in src for k in
      ['set_feature_stats', 'init_output_bias', 'self.float()']))


Cloning into '/content/FakeNewsCredibilityAssesor'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 54 (delta 4), reused 37 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 117.53 KiB | 13.06 MiB/s, done.
Resolving deltas: 100% (4/4), done.

Now in: /content/FakeNewsCredibilityAssesor
Latest commit:
a630661 (grafted, HEAD -> main, origin/main, origin/HEAD) colab_train_from_git: cd to /content before rm -rf (fix getcwd crash)
phase5 fix present: True


## 3 · Install dependencies + NLTK data


In [4]:
# requirements.train.txt pulls in requirements.api.txt automatically.
!pip install -q -r requirements.train.txt
import nltk
for pkg in ['stopwords','punkt','punkt_tab','opinion_lexicon']:
    nltk.download(pkg, quiet=True)
print('Dependencies installed.')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 56.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.4/502.4 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 129.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 4 · Phase 1–3 · Build the dataset

Downloads LIAR-2, MultiFC, FEVER, AVeriTeC from HuggingFace and builds
`data/train.csv`, `val.csv`, `test.csv`.

Add `--sample 3000` for a fast smoke run; omit for the full ~90k-row dataset.


In [5]:
import os
if all(os.path.exists(f'data/{s}.csv') for s in ['train','val','test']):
    print('Data already present — skipping.')
else:
    !python credibility_detector_phases123.py 2>&1 | tail -40



  Imbalance ratio: 9999.50× (⚠ high)

───────────────────────────────────────────────────────
  CONTEXT × SENTIMENT INTERACTION (key features)
───────────────────────────────────────────────────────
  context_sentiment_risk              mean=0.059  r=0.0197
  context_adjusted_sentiment          mean=0.009  r=0.1011
  persuasive_context_flag             mean=0.465  r=-0.2847
  context_credibility_prior           mean=0.562  r=0.3313

───────────────────────────────────────────────────────
  MISSING VALUES
───────────────────────────────────────────────────────
  speaker                         2,724  (3.8%)
  speaker_job                    65,621  (92.0%)
  subject                         5,788  (8.1%)
  state_info                     57,583  (80.8%)
  justification                  10,843  (15.2%)
  historical_credibility         53,096  (74.5%)


Generating plots…
  Saved: 01_credibility_distribution.png
  Saved: 02_label_distribution.png
  Saved: 03_credibility_by_dataset.png
  Save

In [6]:
import pandas as pd
for s in ['train','val','test']:
    df = pd.read_csv(f'data/{s}.csv')
    print(f'{s:5} {len(df):>7,} rows   NaN credibility={df["credibility_score"].isna().sum()}')


train  71,308 rows   NaN credibility=0
val     8,940 rows   NaN credibility=0
test    8,950 rows   NaN credibility=0


## 5 · Phase 4 · TF-IDF baseline (MAE benchmark)


In [7]:
!python phase4_baseline_1.py --no-shap


  PHASE 4 — TF-IDF Baseline Model
  Loaded: train=71,308  val=8,940  test=8,950
  Fitting TF-IDF (max=50,000, ngram=(1, 3), min_df=2, max_df=0.9)…
  TF-IDF matrix: train=(71308, 50000), val=(8940, 50000), test=(8950, 50000)
  Appending engineered features…
  Combined matrix: 50,013 features total (TF-IDF + 13 engineered)
  Training RidgeCV regressor…
  Best alpha: 1.0000

  Evaluation:
  [train] MAE=0.2275  RMSE=0.2778  r=0.7194  F1=0.6237
  [val] MAE=0.2912  RMSE=0.3519  r=0.4478  F1=0.4799
  [test] MAE=0.2892  RMSE=0.3502  r=0.4569  F1=0.4832

  Per-dataset test breakdown:
  dataset            n      MAE       F1
  ──────────── ─────── ──────── ────────
  averitec         340   0.3110   0.4904
  fever          4,597   0.3290   0.3962
  liar2          2,281   0.2286   0.4946
  multifc        1,732   0.2590   0.4602

  ★ BENCHMARK VAL MAE = 0.2912
    DeBERTa (Phase 6) must achieve MAE < 0.2912
    Expected DeBERTa improvement: 0.04–0.08 MAE points

  Saved: models/baseline_tfidf.pkl
 

## 6 · Phase 5 · Fine-tune DeBERTa-v3 on GPU  ⏳ (main step)

Watch for `✓ Forward-pass sanity check OK on 'cuda' ...` before Epoch 1.
val_MAE should fall each epoch (target ≈ 0.20–0.24 vs baseline ≈ 0.287),
`Pearson_r` should be a real number, and there should be **no `NaN-skip`**.


In [11]:
!python phase5_deberta.py --train --device cuda



  PHASE 5 — Fine-tuning: microsoft/deberta-v3-base
  Device: cuda | Epochs: 4 | LR: 2e-05 | Batch: 32
  Max seq len: 96 | Frozen layers: 8 | Mixed precision: off
  Prefix: enabled

  Class weights (low/mid/high): {'high': 0.79, 'low': 1.017, 'mid': 1.333}
Loading tokenizer…
Loading model…
Loading weights: 100% 198/198 [00:00<00:00, 37058.11it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEX

In [12]:
import json
r = json.load(open('models/deberta_results.json'))
print('Best epoch   :', r['best_epoch'])
print('Best val MAE :', r['best_val_MAE'])
print('Test MAE     :', r['test']['MAE'])
print('Test F1      :', r['test']['Macro_F1'])
print('Baseline MAE :', r['baseline_MAE'])
print('Improvement  :', r['improvement'])
print('Per-dataset  :', json.dumps(r.get('per_dataset', {}), indent=2))


Best epoch   : 4
Best val MAE : 0.2553
Test MAE     : 0.2512
Test F1      : 0.5548
Baseline MAE : 0.2912
Improvement  : 0.0359
Per-dataset  : {
  "averitec": {
    "n": 340,
    "MAE": 0.2662,
    "Macro_F1": 0.5038
  },
  "fever": {
    "n": 4597,
    "MAE": 0.2807,
    "Macro_F1": 0.49
  },
  "liar2": {
    "n": 2281,
    "MAE": 0.2029,
    "Macro_F1": 0.5339
  },
  "multifc": {
    "n": 1732,
    "MAE": 0.2335,
    "Macro_F1": 0.5376
  }
}


## 7 · Phase 5b · Calibration & uncertainty


In [ ]:
!python phase5b_calibration.py --device cuda


  PHASE 5b — Calibration & Uncertainty Quantification

Loading Phase 5 module and model…
Loading weights: 100% 198/198 [00:00<00:00, 37006.92it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UN

## 8 · Phase 6 · Speaker profiles + context encoder


In [ ]:
!python phase6_speaker_profiler.py
!python context_encoder.py --visualise


## 9 · Phase 7 · Token-level SHAP explanations


In [ ]:
!python phase7_shap_explainer.py --n-examples 12 --n-global 150


## 10 · (Optional) Mistral-7B QLoRA justification model

**Needs an A100 / high-RAM runtime.** Skip on a plain T4.


In [ ]:
# !python llm_finetune.py --train
print('Uncomment above on an A100 runtime.')


## 11 · Save trained artifacts to Google Drive

So the weights survive the runtime disconnecting.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
dst = '/content/drive/MyDrive/FakeNews_models'
os.makedirs(dst, exist_ok=True)
for f in ['deberta_best.pt','deberta_results.json','deberta_tokenizer',
          'baseline_tfidf.pkl','baseline_results.json','calibration.json',
          'speaker_profiles.json']:
    src = f'models/{f}'
    if os.path.exists(src):
        d = f'{dst}/{f}'
        if os.path.isdir(src):
            shutil.rmtree(d, ignore_errors=True); shutil.copytree(src, d)
        else:
            shutil.copy(src, d)
        print('saved', f)
print('Done → Drive/MyDrive/FakeNews_models')


## 🔄 Re-pull the latest code (run anytime a fix is pushed)

Updates the code from GitHub **without** deleting your `data/` or `models/`
(those are git-ignored, so `git pull` leaves them untouched). Re-run the
phase cells afterwards.


In [10]:
import os
os.chdir('/content/FakeNewsCredibilityAssesor')
!git pull --ff-only
!git log --oneline -1


Already up to date.
55228d8 (HEAD -> main, origin/main, origin/HEAD) Phase 5: ~5-6x faster training (config + mixed precision)


## 12 · Quick inference smoke test


In [ ]:
!python phase5_deberta.py --predict "The unemployment rate fell to a record low." \
    --speaker "Joe Biden" --context "a speech" --device cuda
